# FUNSD benchmark: page read vs page read + crop re-read

Runs the handwritten route on `nielsr/funsd` test pages in two modes and scores both against the labelled **answers**:

| Mode | What runs |
| --- | --- |
| `page_read` | Qwen3-VL-4B reads the whole page (the PRD v1.5 route) |
| `page_read+crop` | the same, then OpenCV box snapping and a cropped re-read of low-confidence and ID fields |

Set **Internet: ON** and **GPU T4**, then run the four code cells in order. Change `PAGES` in cell 3 (the test split has 50 pages; 10 is a quick check).

In [ ]:
# 1. Get the code, install what is missing (first run takes a few minutes)
import os, subprocess
os.chdir("/kaggle/working")
REPO = "/kaggle/working/DOC_Extractor_QUAPT"
BRANCH = "feat/page-read"
URL = "https://github.com/dharicharan/DOC_Extractor_QUAPT.git"
if not os.path.isdir(REPO + "/.git"):
    r = subprocess.run(["git", "clone", "-b", BRANCH, URL, REPO], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("Clone failed (Internet ON?): " + r.stderr)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH])
    subprocess.run(["git", "-C", REPO, "checkout", "-B", BRANCH, "origin/" + BRANCH])
print("commit:", subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
try:
    import transformers, qwen_vl_utils, bitsandbytes, cv2, datasets, doc_extractor
except Exception:
    get_ipython().system("pip install -q uv")
    get_ipython().system(f"cd {REPO} && uv pip install --system -r requirements.txt && uv pip install --system -e .")
    get_ipython().system("uv pip install --system qwen-vl-utils transformers accelerate bitsandbytes datasets")
os.environ.setdefault("DOC_MAX_IMAGE_PIXELS", "2000000")

In [ ]:
# 2. Scoring helpers: gold answers from the FUNSD labels, text and box matching
import re
from difflib import SequenceMatcher
from doc_extractor.crop_verify import _as_box, _leaves
from doc_extractor.vlm_grounding import _null_word_to_none


def norm(s):
    return re.sub(r"[^0-9a-z]", "", str(s).lower())


def gold_answers(words, boxes, tags):
    """ANSWER entities (tags 5 = B-ANSWER, 6 = I-ANSWER): consecutive words joined, boxes unioned."""
    out, cur = [], None
    for w, b, t in zip(words, boxes, tags):
        if t == 5 or (t == 6 and cur is None):
            if cur:
                out.append(cur)
            cur = {"w": [w], "b": list(b)}
        elif t == 6:
            cur["w"].append(w)
            cur["b"] = [min(cur["b"][0], b[0]), min(cur["b"][1], b[1]), max(cur["b"][2], b[2]), max(cur["b"][3], b[3])]
        else:
            if cur:
                out.append(cur)
            cur = None
    if cur:
        out.append(cur)
    return [{"text": " ".join(a["w"]), "box": tuple(map(float, a["b"]))} for a in out if norm(" ".join(a["w"]))]


def same_text(p, g):
    p, g = norm(p), norm(g)
    if not p or not g:
        return False
    return p == g or (len(g) >= 4 and g in p) or SequenceMatcher(None, p, g).ratio() >= 0.8


def iou(a, b):
    if not a or not b:
        return 0.0
    inter = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def predicted(raw):
    out = []
    for path, leaf in _leaves(raw):
        v = _null_word_to_none(leaf.get("value"))
        if v is None or isinstance(v, (dict, list)) or not norm(v):
            continue
        out.append({"value": str(v), "box": _as_box(leaf.get("bbox")), "flagged": leaf.get("crop_verified") is False})
    return out


def score(preds, golds):
    pairs = sorted(((SequenceMatcher(None, norm(p["value"]), norm(g["text"])).ratio(), i, j)
                    for i, p in enumerate(preds) for j, g in enumerate(golds) if same_text(p["value"], g["text"])), reverse=True)
    up, ug, m = set(), set(), []
    for _, i, j in pairs:
        if i not in up and j not in ug:
            up.add(i)
            ug.add(j)
            m.append((preds[i], golds[j]))
    ious = [iou(p["box"], g["box"]) for p, g in m]
    return {"gold": len(golds), "pred": len(preds), "matched": len(m), "box_hits": sum(v >= 0.5 for v in ious),
            "iou": sum(ious), "flagged": sum(p["flagged"] for p in preds),
            "flagged_wrong": sum(p["flagged"] and k not in up for k, p in enumerate(preds))}

In [ ]:
# 3. Run both modes on FUNSD test pages (the model loads once)
import io, tempfile, time, json
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from doc_extractor import crop_verify
from doc_extractor.adapters import VLMAdapter
from doc_extractor.routing import DEFAULT_MODEL, DocumentCategory, RoutedExtractor, build_generic_prompt

PAGES = 10          # the test split has 50 pages
MODES = {"page_read": "0", "page_read+crop": "1"}

ds = load_dataset("nielsr/funsd", split="test")
rows = [ds[i] for i in range(min(PAGES, len(ds)))]
adapter = VLMAdapter(RoutedExtractor().get_extractor(DEFAULT_MODEL))
adapter.crop_verify_allowed = True
prompt = build_generic_prompt(DocumentCategory.HANDWRITTEN)
tmp = Path(tempfile.mkdtemp())
paths = []
for r in rows:
    p = tmp / f"funsd_{r['id']}.png"
    r["image"].convert("RGB").save(p)
    paths.append(p)

results = {}
for mode, flag in MODES.items():
    os.environ["DOC_CROP_VERIFY"] = flag
    per = []
    for r, p in zip(rows, paths):
        golds = gold_answers(r["words"], r["bboxes"], r["ner_tags"])
        t0 = time.perf_counter()
        try:
            raw, _, meta = adapter.extract(str(p), prompt)
        except Exception as e:
            print(mode, r["id"], "FAILED", type(e).__name__, e)
            raw = {}
        s = score(predicted(raw), golds)
        s["sec"] = time.perf_counter() - t0
        per.append(s)
        print(f"[{mode}] page {r['id']}: {s['matched']}/{s['gold']} answers, {s['pred']} values, {s['box_hits']} boxes hit, {s['sec']:.0f}s")
    results[mode] = per
    crop_verify.begin_run()

In [ ]:
# 4. Result table
def summary(per):
    t = {k: sum(p[k] for p in per) for k in per[0]}
    return {
        "answer recall": round(t["matched"] / t["gold"], 3),
        "value precision": round(t["matched"] / max(t["pred"], 1), 3),
        "box hit rate (IoU>=0.5)": round(t["box_hits"] / max(t["matched"], 1), 3),
        "mean box IoU": round(t["iou"] / max(t["matched"], 1), 3),
        "fields flagged by crop check": t["flagged"],
        "flag precision (flagged were wrong)": round(t["flagged_wrong"] / t["flagged"], 3) if t["flagged"] else "-",
        "seconds per page": round(t["sec"] / len(per), 1),
    }


table = pd.DataFrame({m: summary(per) for m, per in results.items()})
display(table)
json.dump(results, open("/kaggle/working/funsd_results.json", "w"), indent=1)
print("Saved /kaggle/working/funsd_results.json")

## Reading the table
- **answer recall:** labelled answers found among the extracted values. **value precision:** extracted values that are real answers.
- **box hit rate and mean IoU:** how well each box sits on its answer (OpenCV snapping should raise these).
- **flag precision:** of the fields the crop check flagged, the share that really were wrong (crop mode only).
- FUNSD pages are typed or printed forms, not handwriting, so use it to check recall and boxes. Judge handwriting on your own forms.